In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

In [ ]:
from pathlib import Path

BASE = Path(
    r"C:\Users\herre\OneDrive\Desktop\Proyectos DATA\SPRINT 13"
)

RUTA_ARTICULOS_PRENSA = (
    BASE
    / "01_data"
    / "02_processed"
    / "prensa"
    / "articulos_con_texto"
)

RUTA_RAW_MENEAME = (
    BASE
    / "01_data"
    / "01_raw"
    / "rrss"
    / "meneame"
)

RUTA_SCRIPTS_MENEAME = (
    BASE
    / "scripts"
    / "meneame"
)

RUTA_RAW_MENEAME.mkdir(
    parents=True,
    exist_ok=True
)

In [2]:
url = (
    "https://www.meneame.net/story/"
    "pp-vox-recortan-10-000-euros-cruz-roja-valenciana-hacer-estudios"
)

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/150.0 Safari/537.36"
    )
}


#Comprobamos que Meneame deja descargar la página
respuesta = requests.get(
    url,
    headers=headers,
    timeout=20
)

print(respuesta.status_code)
print(len(respuesta.text))

200
54971


In [3]:
#Convierte el HTML en un objeto navegable.

soup = BeautifulSoup(
    respuesta.text,
    "html.parser"
)

#Extraemos todo el texto de la página

texto_pagina = soup.get_text(
    " ",
    strip=True
)

print(
    "Qué mala baba" in texto_pagina
)

True


In [4]:
#Busca todo los objetos de tipo comentario.
comentarios_posibles = soup.select(
    '.comment, .comment-body, .comment-text, [id^="comment-"]'
)

print(len(comentarios_posibles))

39


In [5]:
 
for elemento in comentarios_posibles[:10]:
    print(elemento.name, elemento.get('class'), elemento.get('id'))
    print(elemento.get_text(" ", strip=True)[:300])
    print("-" * 80)

div ['comment'] c-2
#2 Robus Queda claro que para esta gente es más importante la propaganda que la salud. Si los votas eres complice. 6 K 84
--------------------------------------------------------------------------------
div ['comment-body'] None
#2 Robus Queda claro que para esta gente es más importante la propaganda que la salud. Si los votas eres complice.
--------------------------------------------------------------------------------
div ['comment-text'] cid-45124623
Queda claro que para esta gente es más importante la propaganda que la salud. Si los votas eres complice.
--------------------------------------------------------------------------------
div ['comment'] c-1
#1 ingenierodepalillos * Qué mala baba, si les quitamos los pocos medios que tienen a los que peor están, no les quedará más remedio que delinquir. Como de los que peor están, la mayoría son de fuera, nuestras flamantes e inservibles estadísticas venderán mejor nuestro libro. Que alguien me corrija,
-------------

In [6]:
from datetime import datetime

comentarios = []

for comentario in soup.select('div.comment'):

    numero = comentario.get('id')

    usuario_elemento = comentario.select_one('a.username')
    usuario = (
        usuario_elemento.get_text(strip=True)
        if usuario_elemento
        else None
    )

    texto_elemento = comentario.select_one('div.comment-text')
    texto = (
        texto_elemento.get_text(" ", strip=True)
        if texto_elemento
        else None
    )

    fecha_elemento = comentario.select_one('span.comment-date')
    timestamp = (
        fecha_elemento.get('data-ts')
        if fecha_elemento
        else None
    )

    fecha = (
        datetime.fromtimestamp(int(timestamp))
        if timestamp
        else None
    )

    votos_elemento = comentario.select_one(
        'div.comment-footer a.votes-counter'
    )
    votos = (
        votos_elemento.get_text(strip=True)
        if votos_elemento
        else None
    )

    karma_elemento = comentario.select_one(
        'div.comment-footer span.votes-counter'
    )
    karma = (
        karma_elemento.get_text(" ", strip=True)
        .replace("K", "")
        .strip()
        if karma_elemento
        else None
    )

    comentarios.append({
        'numero_comentario': numero,
        'usuario': usuario,
        'fecha_comentario': fecha,
        'texto': texto,
        'votos': votos,
        'karma': karma
    })

In [7]:
df_comentarios = pd.DataFrame(comentarios)

df_comentarios

,numero_comentario,usuario,fecha_comentario,texto,votos,karma
0,c-2,Robus,2026-07-16 10:23:48,Queda claro que para esta gente es más importa...,6,84
1,c-1,ingenierodepalillos,2026-07-16 10:18:51,"Qué mala baba, si les quitamos los pocos medio...",2,31
2,c-5,reivaj01,2026-07-16 10:36:16,"#1 No te quito la razón, que te la otorgo al 1...",None,0
3,c-8,ingenierodepalillos,2026-07-16 11:13:22,"#5 No conozco los despachos, he visto trabajad...",1,21
4,c-10,reivaj01,2026-07-16 11:32:44,"#8 No es un ataque personal a la Cruz Roja, es...",1,22
5,c-13,ingenierodepalillos,2026-07-16 14:33:11,"#10 Como le digo, no conozco los detalles de l...",None,0
6,c-6,plutanasio,2026-07-16 11:05:52,Y por eso hace una década que dejé de colabora...,1,21
7,c-3,tierramar,2026-07-16 10:27:53,No vaya a ser qu se descubra que toda la inmig...,None,0
8,c-7,zuppo,2026-07-16 11:09:52,"#3 Pero esos son enchufes buenos, no como el d...",None,0
9,c-4,chavi,2026-07-16 10:33:16,"Lo previsto, vamos. La mayoría de los valencia...",None,0


In [8]:
df_comentarios['votos'] = pd.to_numeric(
    df_comentarios['votos'],
    errors='coerce'
).fillna(0).astype(int)

df_comentarios['karma'] = pd.to_numeric(
    df_comentarios['karma'],
    errors='coerce'
).fillna(0).astype(int)

In [9]:
df_comentarios.dtypes

numero_comentario            object
usuario                      object
fecha_comentario     datetime64[ns]
texto                        object
votos                         int64
karma                         int64
dtype: object

In [10]:
import re

def extraer_respuesta(texto):
    coincidencia = re.match(r'#(\d+)', texto)

    if coincidencia:
        return f"c-{coincidencia.group(1)}"

    return None


df_comentarios['responde_a'] = (
    df_comentarios['texto']
    .apply(extraer_respuesta)
)

df_comentarios[
    ['numero_comentario', 'responde_a', 'texto']
]

,numero_comentario,responde_a,texto
0,c-2,None,Queda claro que para esta gente es más importa...
1,c-1,None,"Qué mala baba, si les quitamos los pocos medio..."
2,c-5,c-1,"#1 No te quito la razón, que te la otorgo al 1..."
3,c-8,c-5,"#5 No conozco los despachos, he visto trabajad..."
4,c-10,c-8,"#8 No es un ataque personal a la Cruz Roja, es..."
5,c-13,c-10,"#10 Como le digo, no conozco los detalles de l..."
6,c-6,None,Y por eso hace una década que dejé de colabora...
7,c-3,None,No vaya a ser qu se descubra que toda la inmig...
8,c-7,c-3,"#3 Pero esos son enchufes buenos, no como el d..."
9,c-4,None,"Lo previsto, vamos. La mayoría de los valencia..."


In [ ]:
df_comentarios.to_csv(
    RUTA_RAW_MENEAME / "meneame_comentarios_prueba.csv",
    index=False,
    encoding="utf-8-sig"
)

# añadir los datos de la noticia a cada comentario.

In [12]:
print(soup.title.get_text(strip=True))

PP y Vox recortan 10.000 euros a Cruz Roja valenciana para hacer estudios que asocian inmigración y delincuencia


Exploramos las etiquetas de cada noticia

In [13]:
for enlace in soup.select('a')[:50]:
    texto = enlace.get_text(" ", strip=True)
    href = enlace.get('href')

    if texto or href:
        print(texto[:80], '->', href)

main action -> #
 -> /
edición general -> /
login -> /login?return=%2Fstory%2Fpp-vox-recortan-10-000-euros-cruz-roja-valenciana-hacer-estudios
registrarse -> /register
comunidades -> /subs
fisgona -> /sneak
nótame -> /notame/
galería -> javascript:fancybox_gallery('all');
ayuda -> https://github.com/Meneame/meneame.net/wiki/Comenzando
publicar -> /submit
Crear artículo -> /submit?type=article&write=true
publicaciones -> /
nuevas -> /queue
mírame -> /mirame
artículos -> /articles
asómate -> /asomate
ayuda -> /ayuda
MÁS -> #
todas -> /
actualidad -> /m/actualidad
ciencia -> /m/ciencia
cultura -> /m/cultura
ocio -> /m/ocio
politica -> /m/politica
tecnología -> /m/tecnología
m/* -> /?meta=_*
RSS -> /rss
 -> https://blog.meneame.net/suscripciones/
más visitadas -> /top_visited
Un divertido viaje a la boda -> /story/divertido-viaje-boda
Impresionantes fotos vintage de Gina Lollobrigida en el rodaje de 'La ley' (1959 -> /story/impresionantes-fotos-vintage-gina-lollobrigida-rodaje-ley-1959
Por

In [14]:
import requests

id_noticia = 4197539

url_api = (
    f"https://www.meneame.net/api/list.php?id={id_noticia}"
)

respuesta_api = requests.get(
    url_api,
    headers=headers,
    timeout=20
)

print(respuesta_api.status_code)
print(respuesta_api.text[:1000])

503



In [ ]:
if respuesta_api.status_code == 200:
    datos_api = respuesta_api.json()

    comentarios_api = datos_api.get(
        "objects",
        []
    )

    df_comentarios_api = pd.DataFrame(
        comentarios_api
    )

    display(df_comentarios_api.head())

else:
    datos_api = None
    comentarios_api = []
    df_comentarios_api = pd.DataFrame()

    print(
        f"Error {respuesta_api.status_code}: "
        "la API no devolvió JSON."
    )

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
if not df_comentarios_api.empty:

    df_comentarios_api[
        "fecha_comentario"
    ] = pd.to_datetime(
        df_comentarios_api["date"].astype(int),
        unit="s"
    )

    df_comentarios_api = (
        df_comentarios_api.rename(
            columns={
                "id": "id_comentario",
                "order": "numero_comentario",
                "user": "usuario",
                "content": "texto",
                "votes": "votos"
            }
        )
    )

    display(df_comentarios_api.head())

else:
    print(
        "No hay comentarios para transformar."
    )

## Buscamos noticias de nuestro data set de prensa que se hayan publicado en Menéame para ver sus comentarios.

In [ ]:
dist_inmigracion2015 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "inmigracion_2015_con_texto.csv"
)

dist_inmigracion2016 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "inmigracion_2016_con_texto.csv"
)

dist_inmigracion2019 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "inmigracion_2019_con_texto.csv"
)

dist_inmigracion2023 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "inmigracion_2023_con_texto.csv"
)

In [ ]:
dist_inmigracion = pd.concat(
    [
        dist_inmigracion2015,
        dist_inmigracion2016,
        dist_inmigracion2019,
        dist_inmigracion2023
    ],
    ignore_index=True
)

print(dist_inmigracion.shape)

dist_inmigracion['anio'].value_counts().sort_index()

(71470, 10)


anio
2015    14333
2016    15318
2019    19546
2023    22273
Name: count, dtype: int64

In [ ]:
dist_inmigracion.duplicated(subset='id').sum()

np.int64(500)

In [ ]:
duplicados_id = dist_inmigracion[
    dist_inmigracion.duplicated(
        subset='id',
        keep=False
    )
].sort_values('id')

print(duplicados_id.shape)

duplicados_id[
    ['id', 'anio', 'publish_date', 'media_name', 'title', 'url']
].head(20)

(1000, 10)


,id,anio,publish_date,media_name,title,url
42718,005d06fc91cf7417da5ad702141ccbbfd189ea42887d93...,2019,2019-08-27,laverdad.es,Alrededor de 60 migrantes son rescatados despu...,https://www.laverdad.es/internacional/alrededo...
42218,005d06fc91cf7417da5ad702141ccbbfd189ea42887d93...,2019,2019-08-27,laverdad.es,Alrededor de 60 migrantes son rescatados despu...,https://www.laverdad.es/internacional/alrededo...
42392,00b6ba192087c61383379ddf450e117809c91342ed7fbb...,2019,2019-08-30,diariosur.es,Los 15 migrantes rescatados del Open Arms lleg...,https://www.diariosur.es/sociedad/migrantes-re...
42892,00b6ba192087c61383379ddf450e117809c91342ed7fbb...,2019,2019-08-30,diariosur.es,Los 15 migrantes rescatados del Open Arms lleg...,https://www.diariosur.es/sociedad/migrantes-re...
42258,00d27347cde7ba44fde31ae2c8fcbe6280c82d09d13917...,2019,2019-08-28,eleconomista.es,El Audaz llegará el viernes a Algeciras con lo...,//www.eleconomista.es/politica/noticias/100578...
42758,00d27347cde7ba44fde31ae2c8fcbe6280c82d09d13917...,2019,2019-08-28,eleconomista.es,El Audaz llegará el viernes a Algeciras con lo...,//www.eleconomista.es/politica/noticias/100578...
42270,0118e24fd020cb97d972aa7292134d73ed801f33fd86a8...,2019,2019-08-28,latribunadecuenca.es,Hungría exige a Europa que pague su valla anti...,https://www.latribunadecuenca.es/noticia/Z7BAA...
42770,0118e24fd020cb97d972aa7292134d73ed801f33fd86a8...,2019,2019-08-28,latribunadecuenca.es,Hungría exige a Europa que pague su valla anti...,https://www.latribunadecuenca.es/noticia/Z7BAA...
43031,02d8e5f524eab42e6d5fc57f1ab971531e5acf39fa580f...,2019,2019-08-30,diariodeavila.es,Más de 150 inmigrantes entran en Ceuta en un s...,https://www.diariodeavila.es/noticia/ZD06A6AAD...
42531,02d8e5f524eab42e6d5fc57f1ab971531e5acf39fa580f...,2019,2019-08-30,diariodeavila.es,Más de 150 inmigrantes entran en Ceuta en un s...,https://www.diariodeavila.es/noticia/ZD06A6AAD...


In [ ]:
dist_inmigracion.duplicated().sum()

np.int64(470)

In [ ]:
dist_inmigracion = (
    dist_inmigracion
    .drop_duplicates()
    .reset_index(drop=True)
)

print(dist_inmigracion.shape)
print(dist_inmigracion.duplicated(subset='id').sum())

(71000, 10)
30


In [ ]:
duplicados_restantes = dist_inmigracion[
    dist_inmigracion.duplicated(
        subset='id',
        keep=False
    )
].sort_values('id')

duplicados_restantes[
    ['id', 'anio', 'publish_date', 'media_name', 'title', 'url', 'texto']
]

,id,anio,publish_date,media_name,title,url,texto
42670,1097d3bd7ffaa80244342b13885454e613acf583b8a91c...,2019,2019-08-29,20minutos.com,EEUU: Paperas enferman a centenares de inmigra...,https://www.20minutos.com/noticia/250707/0/eeu...,Agencia EFE\n \n ...
42348,1097d3bd7ffaa80244342b13885454e613acf583b8a91c...,2019,2019-08-29,20minutos.com,EEUU: Paperas enferman a centenares de inmigra...,https://www.20minutos.com/noticia/250707/0/eeu...,NaN
42674,1392585852ff1ec1ea1e315afaf1fabab515c6f55ca35b...,2019,2019-08-30,elperiodicoextremadura.com,153 migrantes saltan la valla en Ceuta en la p...,https://www.elperiodicoextremadura.com/noticia...,EUROPA PRESS\n153 personas de origen subsahari...
42464,1392585852ff1ec1ea1e315afaf1fabab515c6f55ca35b...,2019,2019-08-30,elperiodicoextremadura.com,153 migrantes saltan la valla en Ceuta en la p...,https://www.elperiodicoextremadura.com/noticia...,EUROPA PRESS\n153 personas de origen subsahari...
42532,14290d4e27daebae7e2f5e80ff7c8833a62f1e8951ecc2...,2019,2019-08-30,elimparcial.es,El Gobierno solo devuelve a Marruecos a siete ...,https://www.elimparcial.es/noticia/204399/naci...,NaN
42676,14290d4e27daebae7e2f5e80ff7c8833a62f1e8951ecc2...,2019,2019-08-30,elimparcial.es,El Gobierno solo devuelve a Marruecos a siete ...,https://www.elimparcial.es/noticia/204399/naci...,"(�/�X��/S�KKf�����(�O�""�����F���""W�J�O!M`MX�35..."
42320,1515a975e4db9f88597808044d5e0082435677829a423d...,2019,2019-08-29,laopiniondemalaga.es,El CATE del puerto acoge a los primeros 131 in...,https://www.laopiniondemalaga.es/malaga/2019/0...,E. Press\nEl Centro de Atención Temporal de Ex...
42666,1515a975e4db9f88597808044d5e0082435677829a423d...,2019,2019-08-29,laopiniondemalaga.es,El CATE del puerto acoge a los primeros 131 in...,https://www.laopiniondemalaga.es/malaga/2019/0...,E. Press\nEl Centro de Atención Temporal de Ex...
42653,1535b5553d9739fd99a6b02389ce0969ac1eccb603cc69...,2019,2019-08-27,laopiniondemalaga.es,El CAED de MÃ¡laga tiene 230 plazas para una p...,https://www.laopiniondemalaga.es/malaga/2019/0...,Cristóbal G. Montilla\nLa llegada de personas ...
42178,1535b5553d9739fd99a6b02389ce0969ac1eccb603cc69...,2019,2019-08-27,laopiniondemalaga.es,El CAED de MÃ¡laga tiene 230 plazas para una p...,https://www.laopiniondemalaga.es/malaga/2019/0...,Cristóbal G. Montilla\nLa llegada de personas ...


In [ ]:
dist_inmigracion['longitud_texto'] = (
    dist_inmigracion['texto']
    .fillna('')
    .str.len()
)

dist_inmigracion = (
    dist_inmigracion
    .sort_values('longitud_texto', ascending=False)
    .drop_duplicates(subset='id', keep='first')
    .drop(columns='longitud_texto')
    .reset_index(drop=True)
)

In [ ]:
dist_inmigracion.duplicated(subset='id').sum()

np.int64(0)

In [ ]:
noticia_prueba = dist_inmigracion[
    dist_inmigracion['url'].notna()
].sample(1, random_state=42).iloc[0]

print(noticia_prueba['title'])
print(noticia_prueba['url'])
print(noticia_prueba['anio'])

El domingo llega el primer vuelo con 19 refugiados procedentes de Italia
http://www.europasur.es/article/espana/2148957/domingo/llega/primer/vuelo/con/refugiados/procedentes/italia.html
2015


In [ ]:
from urllib.parse import quote
import requests
from bs4 import BeautifulSoup

url_original = "http://www.europasur.es/article/espana/2148957/domingo/llega/primer/vuelo/con/refugiados/procedentes/italia.html"

url_busqueda = (
    f"https://www.meneame.net/search?q={quote(url_original)}"
)

respuesta = requests.get(
    url_busqueda,
    headers=headers,
    timeout=20
)

print(respuesta.status_code)
print(respuesta.url)

200
https://www.meneame.net/search?q=http%3A//www.europasur.es/article/espana/2148957/domingo/llega/primer/vuelo/con/refugiados/procedentes/italia.html


In [ ]:
soup_busqueda = BeautifulSoup(
    respuesta.text,
    "html.parser"
)

for enlace in soup_busqueda.select('a[href^="/story/"]')[:20]:
    print(
        enlace.get_text(" ", strip=True),
        "->",
        enlace.get("href")
    )

In [ ]:
print(soup_busqueda.title)

<title>búsqueda de «http://www.europasur.es/article/espana/2148957/domingo/llega/primer/vuelo/con/refugiados/procedentes/italia.html»</title>


In [ ]:
texto = soup_busqueda.get_text(" ", strip=True)

print(texto[:1000])

búsqueda de «http://www.europasur.es/article/espana/2148957/domingo/llega/primer/vuelo/con/refugiados/procedentes/italia.html» · main action × edición general login registrarse comunidades fisgona nótame galería ayuda publicar Crear artículo publicaciones nuevas mírame artículos asómate ayuda suscripciones por RSS búsqueda: http://www.europasur.es/article/espana/2148957/domingo/llega/primer/vuelo/con/refugiados/procedentes/italia.html publicadas en cola todos los comentarios links posts comments campos... url tags title site todo el texto estado... published queued discard autodiscard abuse todas período... 24 horas 48 horas última semana último mes 6 meses 1 año todas por relevancia por fecha usuario: encontrados: 0, tiempo total: 0.006 segundos suscripciones por RSS publicadas en cola más activas todos los comentarios ayuda faq ayuda wiki avisar errores avisar abusos +menéame tarifas publicitarias novedades tendencias síguenos en twitter nótame blog estadísticas populares más comenta

In [ ]:
candidatos = (
    dist_inmigracion[
        dist_inmigracion['media_name'].isin([
            'eldiario.es',
            'publico.es',
            'elpais.com',
            'elmundo.es',
            'abc.es',
            '20minutos.es'
        ])
    ]
    .sample(20, random_state=42)
)

candidatos[['media_name', 'title', 'url', 'anio']]

,media_name,title,url,anio
27654,eldiario.es,"Sephora, la bebé de 13 meses que murió al caer...",https://www.eldiario.es/canariasahora/sociedad...,2019
20663,elpais.com,Jude Law acude en ayuda de los refugiados en C...,http://elpais.com/elpais/2016/02/22/estilo/145...,2016
6790,abc.es,El gobierno alemán expresa ahora «reticencias»...,https://www.abc.es/internacional/gobierno-alem...,2023
18126,elpais.com,Al menos 25 migrantes centroamericanos mueren ...,https://elpais.com/internacional/2019/03/08/me...,2019
23052,publico.es,El Consejo de Europa da un toque de atención a...,http://www.publico.es/sociedad/consejo-europa-...,2016
66541,abc.es,Casi nueve horas en el aire para dar con una p...,https://sevilla.abc.es/espana/canarias/abci-ca...,2019
1264,publico.es,Camareros migrantes 'vs' condiciones de trabaj...,http://www.publico.es/politica/camareros-migra...,2023
37507,abc.es,Cae el Gobierno de Rutte en los Países Bajos p...,https://www.abc.es/internacional/cae-gobierno-...,2023
24386,eldiario.es,El Gobierno de Canarias y Fiscalía acuerdan un...,https://www.eldiario.es/canariasahora/migracio...,2023
14682,elpais.com,Especial sobre la tragedia de los refugiados e...,http://elpais.com/cultura/2016/02/17/televisio...,2016


In [ ]:
from urllib.parse import quote
from bs4 import BeautifulSoup
import time

def buscar_url_en_meneame(url_original):
    url_busqueda = (
        f"https://www.meneame.net/search?q={quote(url_original)}"
    )

    respuesta = requests.get(
        url_busqueda,
        headers=headers,
        timeout=20
    )

    soup = BeautifulSoup(respuesta.text, "html.parser")
    texto = soup.get_text(" ", strip=True)

    if "encontrados: 0" in texto:
        return 0

    resultados = soup.select('a[href^="/story/"]')
    return len(resultados)

In [ ]:
resultados_prueba = []

for _, fila in candidatos.iterrows():

    encontrados = buscar_url_en_meneame(fila['url'])

    resultados_prueba.append({
        'titulo': fila['title'],
        'medio': fila['media_name'],
        'anio': fila['anio'],
        'url': fila['url'],
        'encontrados': encontrados
    })

    time.sleep(1)

df_prueba_meneame = pd.DataFrame(resultados_prueba)

df_prueba_meneame

,titulo,medio,anio,url,encontrados
0,"Sephora, la bebé de 13 meses que murió al caer...",eldiario.es,2019,https://www.eldiario.es/canariasahora/sociedad...,0
1,Jude Law acude en ayuda de los refugiados en C...,elpais.com,2016,http://elpais.com/elpais/2016/02/22/estilo/145...,0
2,El gobierno alemán expresa ahora «reticencias»...,abc.es,2023,https://www.abc.es/internacional/gobierno-alem...,0
3,Al menos 25 migrantes centroamericanos mueren ...,elpais.com,2019,https://elpais.com/internacional/2019/03/08/me...,0
4,El Consejo de Europa da un toque de atención a...,publico.es,2016,http://www.publico.es/sociedad/consejo-europa-...,0
5,Casi nueve horas en el aire para dar con una p...,abc.es,2019,https://sevilla.abc.es/espana/canarias/abci-ca...,0
6,Camareros migrantes 'vs' condiciones de trabaj...,publico.es,2023,http://www.publico.es/politica/camareros-migra...,0
7,Cae el Gobierno de Rutte en los Países Bajos p...,abc.es,2023,https://www.abc.es/internacional/cae-gobierno-...,0
8,El Gobierno de Canarias y Fiscalía acuerdan un...,eldiario.es,2023,https://www.eldiario.es/canariasahora/migracio...,0
9,Especial sobre la tragedia de los refugiados e...,elpais.com,2016,http://elpais.com/cultura/2016/02/17/televisio...,0


In [ ]:
coincidencia = df_prueba_meneame[
    df_prueba_meneame['encontrados'] > 0
].iloc[0]

print(coincidencia['titulo'])
print(coincidencia['url'])

Todo lo que se sabe sobre Abdalmasih H., el refugiado sirio de 32 años que ha apuñalado a cuatro niños en un parque de Annecy
https://www.20minutos.es/noticia/5135923/0/sirio-solicitante-asilo-hombre-autor-apunalamiento-ninos-annency-francia/


In [ ]:
url_busqueda = (
    f"https://www.meneame.net/search?q={quote(coincidencia['url'])}"
)

respuesta = requests.get(
    url_busqueda,
    headers=headers,
    timeout=20
)

soup_resultado = BeautifulSoup(
    respuesta.text,
    "html.parser"
)

urls_meneame = {
    enlace.get('href')
    for enlace in soup_resultado.select('a[href^="/story/"]')
}

urls_meneame

{'/story/todo-sabe-sobre-abdalmasih-h-refugiado-sirio-32-anos-ha-cuatro'}

In [ ]:
url_meneame = (
    "https://www.meneame.net"
    "/story/todo-sabe-sobre-abdalmasih-h-refugiado-sirio-32-anos-ha-cuatro"
)

respuesta_noticia = requests.get(
    url_meneame,
    headers=headers,
    timeout=20
)

soup_noticia = BeautifulSoup(
    respuesta_noticia.text,
    "html.parser"
)

print(respuesta_noticia.status_code)

200


In [ ]:
enlace_rss = soup_noticia.select_one(
    'a[href^="/comments_rss?id="]'
)

print(enlace_rss.get('href'))

/comments_rss?id=3821932


In [ ]:
id_noticia = int(
    re.search(
        r'id=(\d+)',
        enlace_rss.get('href')
    ).group(1)
)

print(id_noticia)

3821932


In [ ]:
url_api = (
    f"https://www.meneame.net/api/list.php?id={id_noticia}"
)

respuesta_api = requests.get(
    url_api,
    headers=headers,
    timeout=20
)

if respuesta_api.status_code == 200:

    datos = respuesta_api.json()

    print(
        "Comentarios recuperados:",
        len(datos.get("objects", []))
    )

else:

    datos = None

    print(
        f"Error {respuesta_api.status_code}: "
        "la API no devolvió JSON."
    )

206


In [ ]:
len(dist_inmigracion)

70970

# Extracción del total de noticias

In [ ]:
import time
import requests
import pandas as pd

from bs4 import BeautifulSoup
from pathlib import Path
from urllib.parse import quote, urljoin

In [ ]:
RUTA_RAW = RUTA_RAW_MENEAME

ARCHIVO_PROGRESO_INMIGRACION = (
    RUTA_RAW
    / "busqueda_noticias_meneame.csv"
)

ARCHIVO_ERRORES_INMIGRACION = (
    RUTA_RAW
    / "errores_busqueda_meneame.csv"
)

Buscar noticias en Menéame

In [ ]:
def buscar_noticia_meneame(url_original, sesion, headers):
    if pd.isna(url_original) or not str(url_original).strip():
        return []

    url_busqueda = (
        f"https://www.meneame.net/search?q={quote(str(url_original).strip())}"
    )

    respuesta = sesion.get(
        url_busqueda,
        headers=headers,
        timeout=30
    )

    respuesta.raise_for_status()

    soup = BeautifulSoup(respuesta.text, "html.parser")

    rutas = {
        enlace.get("href")
        for enlace in soup.select('a[href^="/story/"]')
        if enlace.get("href")
    }

    return sorted(
        urljoin("https://www.meneame.net", ruta)
        for ruta in rutas
    )

Preparamos Ejecución

In [ ]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/150.0 Safari/537.36"
    )
}

sesion = requests.Session()

In [ ]:
if ARCHIVO_PROGRESO_INMIGRACION.exists():
    progreso_anterior = pd.read_csv(ARCHIVO_PROGRESO_INMIGRACION)

    ids_procesados = set(
        progreso_anterior["id_prensa"]
        .dropna()
        .astype(str)
    )

    resultados = progreso_anterior.to_dict("records")

    print(f"Noticias recuperadas del progreso: {len(ids_procesados):,}")
else:
    ids_procesados = set()
    resultados = []

errores = []

Noticias recuperadas del progreso: 66,198


Buscamos en la 70.000 noticias

In [ ]:
total = len(dist_inmigracion)

for posicion, fila in enumerate(
    dist_inmigracion.itertuples(index=False),
    start=1
):
    id_prensa = str(fila.id)

    if id_prensa in ids_procesados:
        continue

    try:
        urls_meneame = buscar_noticia_meneame(
            url_original=fila.url,
            sesion=sesion,
            headers=headers
        )

        if urls_meneame:
            for url_meneame in urls_meneame:
                resultados.append({
                    "id_prensa": id_prensa,
                    "anio": fila.anio,
                    "publish_date": fila.publish_date,
                    "media_name": fila.media_name,
                    "titulo_prensa": fila.title,
                    "url_original": fila.url,
                    "encontrada_meneame": True,
                    "url_meneame": url_meneame
                })
        else:
            resultados.append({
                "id_prensa": id_prensa,
                "anio": fila.anio,
                "publish_date": fila.publish_date,
                "media_name": fila.media_name,
                "titulo_prensa": fila.title,
                "url_original": fila.url,
                "encontrada_meneame": False,
                "url_meneame": None
            })

        ids_procesados.add(id_prensa)

    except Exception as error:
        errores.append({
            "id_prensa": id_prensa,
            "url_original": fila.url,
            "error": str(error)
        })


    if posicion % 100 == 0: #guarda cada 100 noticias
        pd.DataFrame(resultados).to_csv(
            ARCHIVO_PROGRESO_INMIGRACION,
            index=False,
            encoding="utf-8-sig"
        )

        pd.DataFrame(errores).to_csv(
            ARCHIVO_ERRORES_INMIGRACION,
            index=False,
            encoding="utf-8-sig"
        )

        encontradas = sum(
            resultado["encontrada_meneame"]
            for resultado in resultados
        )

        print(
            f"{posicion:,}/{total:,} | "
            f"Encontradas: {encontradas:,} | "
            f"Errores: {len(errores):,}"
        )

    # Evita sobrecargar el servidor
    time.sleep(1)

66,300/70,970 | Encontradas: 666 | Errores: 0
66,400/70,970 | Encontradas: 666 | Errores: 0
66,500/70,970 | Encontradas: 666 | Errores: 0
66,600/70,970 | Encontradas: 667 | Errores: 0
66,700/70,970 | Encontradas: 667 | Errores: 0
66,800/70,970 | Encontradas: 668 | Errores: 0
66,900/70,970 | Encontradas: 669 | Errores: 0
67,000/70,970 | Encontradas: 669 | Errores: 0
67,100/70,970 | Encontradas: 670 | Errores: 0
67,200/70,970 | Encontradas: 670 | Errores: 0
67,300/70,970 | Encontradas: 670 | Errores: 0
67,400/70,970 | Encontradas: 671 | Errores: 0
67,500/70,970 | Encontradas: 672 | Errores: 0
67,600/70,970 | Encontradas: 672 | Errores: 0
67,700/70,970 | Encontradas: 672 | Errores: 0
67,800/70,970 | Encontradas: 673 | Errores: 0
67,900/70,970 | Encontradas: 674 | Errores: 0
68,000/70,970 | Encontradas: 674 | Errores: 0
68,100/70,970 | Encontradas: 674 | Errores: 0
68,200/70,970 | Encontradas: 674 | Errores: 0
68,300/70,970 | Encontradas: 674 | Errores: 0
68,400/70,970 | Encontradas: 674 |

Guardado final

In [ ]:
df_busqueda_meneame = pd.DataFrame(resultados)

df_busqueda_meneame.to_csv(
    ARCHIVO_PROGRESO_INMIGRACION,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame(errores).to_csv(
    ARCHIVO_ERRORES_INMIGRACION,
    index=False,
    encoding="utf-8-sig"
)

Resumen

In [ ]:
print("Noticias de prensa:", total)

print(
    "Noticias encontradas en Menéame:",
    df_busqueda_meneame.loc[
        df_busqueda_meneame["encontrada_meneame"],
        "id_prensa"
    ].nunique()
)

print(
    "Publicaciones distintas de Menéame:",
    df_busqueda_meneame["url_meneame"].nunique()
)

print("Errores:", len(errores))

Noticias de prensa: 70970
Noticias encontradas en Menéame: 681
Publicaciones distintas de Menéame: 683
Errores: 0


#LGTBI

In [ ]:
dist_lgtb2015 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "lgtb_2015_con_texto.csv"
)

dist_lgtb2016 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "lgtb_2016_con_texto.csv"
)

dist_lgtb2019 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "lgtb_2019_con_texto.csv"
)

dist_lgtb2023 = pd.read_csv(
    RUTA_ARTICULOS_PRENSA
    / "lgtb_2023_con_texto.csv"
)

In [ ]:
dist_lgtb = pd.concat(
    [
        dist_lgtb2015,
        dist_lgtb2016,
        dist_lgtb2019,
        dist_lgtb2023
    ],
    ignore_index=True
)

print(dist_lgtb.shape)

(22572, 10)


In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

total_lgtb = len(dist_lgtb)

ARCHIVO_PROGRESO_LGTB = (
    RUTA_RAW / "busqueda_noticias_meneame_lgtbi.csv"
)

ARCHIVO_ERRORES_LGTB = (
    RUTA_RAW / "errores_busqueda_meneame_lgtbi.csv"
)


# ============================================================
# RECUPERAR EL PROGRESO ANTERIOR
# ============================================================

if ARCHIVO_PROGRESO_LGTB.exists():

    progreso_lgtb = pd.read_csv(
        ARCHIVO_PROGRESO_LGTB,
        dtype={"id_prensa": str}
    )

    resultados_lgtb = progreso_lgtb.to_dict("records")

    ids_procesados_lgtb = set(
        progreso_lgtb["id_prensa"]
        .dropna()
        .astype(str)
    )

else:

    resultados_lgtb = []
    ids_procesados_lgtb = set()


errores_lgtb = []
nuevas_procesadas_lgtb = 0

pendientes_lgtb = (
    total_lgtb
    - len(ids_procesados_lgtb)
)

print(
    f"Noticias recuperadas: "
    f"{len(ids_procesados_lgtb):,}"
)

print(
    f"Noticias pendientes: "
    f"{pendientes_lgtb:,}"
)


# ============================================================
# FUNCIÓN PARA GUARDAR EL PROGRESO
# ============================================================

def guardar_progreso_lgtb():

    pd.DataFrame(
        resultados_lgtb
    ).to_csv(
        ARCHIVO_PROGRESO_LGTB,
        index=False,
        encoding="utf-8-sig"
    )

    pd.DataFrame(
        errores_lgtb,
        columns=[
            "id_prensa",
            "url_original",
            "error"
        ]
    ).to_csv(
        ARCHIVO_ERRORES_LGTB,
        index=False,
        encoding="utf-8-sig"
    )


# ============================================================
# PROCESAR ÚNICAMENTE LAS NOTICIAS PENDIENTES
# ============================================================

for posicion, fila in enumerate(
    dist_lgtb.itertuples(index=False),
    start=1
):

    id_prensa = str(fila.id)

    # Saltar noticias ya procesadas
    if id_prensa in ids_procesados_lgtb:
        continue

    try:

        urls_meneame = buscar_noticia_meneame(
            url_original=fila.url,
            sesion=sesion,
            headers=headers
        )

        if urls_meneame:

            for url_meneame in urls_meneame:

                resultados_lgtb.append({
                    "id_prensa": id_prensa,
                    "anio": fila.anio,
                    "publish_date": fila.publish_date,
                    "media_name": fila.media_name,
                    "titulo_prensa": fila.title,
                    "url_original": fila.url,
                    "encontrada_meneame": True,
                    "url_meneame": url_meneame
                })

        else:

            resultados_lgtb.append({
                "id_prensa": id_prensa,
                "anio": fila.anio,
                "publish_date": fila.publish_date,
                "media_name": fila.media_name,
                "titulo_prensa": fila.title,
                "url_original": fila.url,
                "encontrada_meneame": False,
                "url_meneame": None
            })

        
        ids_procesados_lgtb.add(id_prensa)
        nuevas_procesadas_lgtb += 1

        
        guardar_progreso_lgtb()

        encontradas_lgtb = sum(
            bool(resultado["encontrada_meneame"])
            for resultado in resultados_lgtb
        )

        pendientes_lgtb = (
            total_lgtb
            - len(ids_procesados_lgtb)
        )

        print(
            f"{len(ids_procesados_lgtb):,}"
            f"/{total_lgtb:,} | "
            f"Nuevas: {nuevas_procesadas_lgtb:,} | "
            f"Pendientes: {pendientes_lgtb:,} | "
            f"Encontradas: {encontradas_lgtb:,} | "
            f"Errores: {len(errores_lgtb):,}"
        )

    except Exception as error:

        errores_lgtb.append({
            "id_prensa": id_prensa,
            "url_original": fila.url,
            "error": str(error)
        })

        # Guardar también el registro del error
        guardar_progreso_lgtb()

        print(
            f"Error en {id_prensa}: {error}"
        )

  
    time.sleep(0.3)


Noticias recuperadas: 22,500
Noticias pendientes: 72
22,501/22,572 | Nuevas: 1 | Pendientes: 71 | Encontradas: 379 | Errores: 0
22,502/22,572 | Nuevas: 2 | Pendientes: 70 | Encontradas: 379 | Errores: 0
22,503/22,572 | Nuevas: 3 | Pendientes: 69 | Encontradas: 379 | Errores: 0
22,504/22,572 | Nuevas: 4 | Pendientes: 68 | Encontradas: 379 | Errores: 0
22,505/22,572 | Nuevas: 5 | Pendientes: 67 | Encontradas: 379 | Errores: 0
22,506/22,572 | Nuevas: 6 | Pendientes: 66 | Encontradas: 379 | Errores: 0
22,507/22,572 | Nuevas: 7 | Pendientes: 65 | Encontradas: 379 | Errores: 0
22,508/22,572 | Nuevas: 8 | Pendientes: 64 | Encontradas: 379 | Errores: 0
22,509/22,572 | Nuevas: 9 | Pendientes: 63 | Encontradas: 379 | Errores: 0
22,510/22,572 | Nuevas: 10 | Pendientes: 62 | Encontradas: 379 | Errores: 0
22,511/22,572 | Nuevas: 11 | Pendientes: 61 | Encontradas: 379 | Errores: 0
22,512/22,572 | Nuevas: 12 | Pendientes: 60 | Encontradas: 379 | Errores: 0
22,513/22,572 | Nuevas: 13 | Pendientes: 59 

In [ ]:
# ============================================================
# GUARDADO Y COMPROBACIÓN FINAL
# ============================================================

guardar_progreso_lgtb()

df_busqueda_meneame_lgtb = pd.DataFrame(
    resultados_lgtb
)

noticias_encontradas_lgtb = (
    df_busqueda_meneame_lgtb.loc[
        df_busqueda_meneame_lgtb[
            "encontrada_meneame"
        ].eq(True),
        "id_prensa"
    ]
    .astype(str)
    .nunique()
)

publicaciones_meneame_lgtb = (
    df_busqueda_meneame_lgtb[
        "url_meneame"
    ].nunique()
)


In [ ]:

print()
print("Proceso LGTBI terminado")
print("------------------------")

print(
    "Noticias de prensa:",
    f"{total_lgtb:,}"
)

print(
    "Noticias procesadas:",
    f"{len(ids_procesados_lgtb):,}"
)

print(
    "Noticias nuevas procesadas:",
    f"{nuevas_procesadas_lgtb:,}"
)

print(
    "Noticias encontradas en Menéame:",
    f"{noticias_encontradas_lgtb:,}"
)

print(
    "Publicaciones distintas de Menéame:",
    f"{publicaciones_meneame_lgtb:,}"
)

print(
    "Errores:",
    f"{len(errores_lgtb):,}"
)


Proceso LGTBI terminado
------------------------
Noticias de prensa: 22,572
Noticias procesadas: 22,572
Noticias nuevas procesadas: 72
Noticias encontradas en Menéame: 375
Publicaciones distintas de Menéame: 379
Errores: 0


## Extracción de los comentarios en las noticias de Menéame

In [ ]:
%run "C:/Users/herre/OneDrive/Desktop/Proyectos DATA/SPRINT 13/scripts/meneame/extraer_comentarios_meneame.py"

Publicaciones únicas: 1,059
Ya completadas: 779
Pendientes: 280
1/280 | no_disponible | comentarios:  | https://www.meneame.net/story/10-bulos-sobre-migrantes-han-intentado-colarte-este-verano
2/280 | no_disponible | comentarios:  | https://www.meneame.net/story/400-inmigrantes-entran-ceuta-ilegalmente-traves-valla
3/280 | error | comentarios:  | https://www.meneame.net/story/400-metros-hasta-siria-regreso-cuerpos-refugiados-fallecidos
4/280 | no_disponible | comentarios:  | https://www.meneame.net/story/66-espanoles-favor-controlar-mejor-fronteras-frente-refugiados
5/280 | error | comentarios:  | https://www.meneame.net/story/abandonados-suelo-comisaria-inmigrantes-graves-deshidratados
6/280 | no_disponible | comentarios:  | https://www.meneame.net/story/abascal-harta-distorsiones-respetamos-gais-inmigrantes-mujeres
7/280 | no_disponible | comentarios:  | https://www.meneame.net/story/abascal-no-intentaria-reconvertir-hijo-homosexual-querria-igual
8/280 | no_disponible | comentarios: 

### Reanudar publicaciones pendientes o con error

El script consulta el archivo de progreso, omite las publicaciones completadas y reintenta únicamente las pendientes.